# Homework 0 - Using k-NN to Analyze Scientific Data

This assignment will be using the elementary KNN classification model to classify data from the penguin dataset. This assignment will focus on performing operations and manipulations on the data to get clear insight.

In [26]:
# Adding all dependencies to the environment
import pandas as pd  # Data manipulation and analysis
import scipy as sc  # Scientific Computing
import numpy as np  # Numerical Computing
import matplotlib.pyplot as plt  # Data Visualization with Static Plots
import plotly.graph_objects as go  # Interative Data Visualization
import math  # Standard Library for Math Operations
from sklearn.model_selection import train_test_split  # Data Splitting
from sklearn.neighbors import KNeighborsClassifier  # ML Models
from sklearn.base import BaseEstimator  # Generalized Model Type-Checking
from sklearn.metrics import accuracy_score

## Problem 1 - Loading the dataset

Loading the penguins dataset (*peguins_size*) into a dataframe.

In [15]:
df = pd.read_csv('data/penguins_size.csv')  # Load the csv file into a pandas dataframe

## Problem 2 - Cleaning the data

Firstly, examining the dataframe.

In [16]:
print(f"Columns:\n{df.dtypes}\n")
print(f"Description:\n{df.describe}")

Columns:
species               object
island                object
culmen_length_mm     float64
culmen_depth_mm      float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object

Description:
<bound method NDFrame.describe of     species     island  culmen_length_mm  culmen_depth_mm  flipper_length_mm  \
0    Adelie  Torgersen              39.1             18.7              181.0   
1    Adelie  Torgersen              39.5             17.4              186.0   
2    Adelie  Torgersen              40.3             18.0              195.0   
3    Adelie  Torgersen               NaN              NaN                NaN   
4    Adelie  Torgersen              36.7             19.3              193.0   
..      ...        ...               ...              ...                ...   
339  Gentoo     Biscoe               NaN              NaN                NaN   
340  Gentoo     Biscoe              46.8             14.3              215.0   
341  G

Removing any incomplete records.

In [17]:
df = df.dropna(how="any")  # Dropping any incomplete data
df

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


Verifying that the new dataframe only contains complete records.

In [18]:
print(df.isna().sum())  # Obtain a count of rows that contain an incomplete record

df  # Display the full dataframe

species              0
island               0
culmen_length_mm     0
culmen_depth_mm      0
flipper_length_mm    0
body_mass_g          0
sex                  0
dtype: int64


,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,MALE
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,FEMALE
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,MALE
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,FEMALE


## Problem 3 - Integer classifications for columns

Create integer categorizations for the original species and sex fields, named species_type and sex_type.

First, determine the unique names for species and sex.

In [19]:
# Obtain the unique names for species and sex
species_names = df['species'].unique()
sex_names = df['sex'].unique()
print(f"Species Names: {species_names}, Sex Names: {sex_names}")  # Quick manual verification

Species Names: ['Adelie' 'Chinstrap' 'Gentoo'], Sex Names: ['MALE' 'FEMALE' '.']


Next, categorize these names into integer representations, add to the df, and convert from floats to ints.

In [20]:

# Create new df columns for species types and sex types
for i in range(len(species_names)):
    df.loc[df['species'] == species_names[i], 'species_type'] = i
for j in range(len(sex_names)):
    df.loc[df['sex'] == sex_names[j], 'sex_type'] = j
    
# Converting the float columns to ints
df = df.astype({'species_type': 'int32', 'sex_type': 'int32'})
    
# Verify the new df with a random sample
df.sample(10)

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex,species_type,sex_type
190,Chinstrap,Dream,46.9,16.6,192.0,2700.0,FEMALE,1,1
221,Gentoo,Biscoe,50.0,16.3,230.0,5700.0,MALE,2,0
225,Gentoo,Biscoe,46.5,13.5,210.0,4550.0,FEMALE,2,1
91,Adelie,Dream,41.1,18.1,205.0,4300.0,MALE,0,0
330,Gentoo,Biscoe,50.5,15.2,216.0,5000.0,FEMALE,2,1
129,Adelie,Torgersen,44.1,18.0,210.0,4000.0,MALE,0,0
203,Chinstrap,Dream,51.4,19.0,201.0,3950.0,MALE,1,0
197,Chinstrap,Dream,50.8,18.5,201.0,4450.0,MALE,1,0
310,Gentoo,Biscoe,47.5,15.0,218.0,4950.0,FEMALE,2,1
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,FEMALE,2,1


## Problem 4 - Feature Selection

First, we should do an initial selection of which features could be used to obtain the best accuracy. 

Using simple domain knowledge, we know that attributes that define geographical or physical features would be beneficial, while attributes such as sex or indexes may not. Therefore, we can start with these attributes.

In [21]:
possible_features = ['island', 'culmen_length_mm', 'culmen_depth_mm', \
                       'flipper_length_mm', 'body_mass_g']  # Initial features to consider

Now that we have the deisred features, we isolate the features and target.

In [22]:
y_df = df['species_type']  # Target
x_df = df.drop(columns=['species_type', 'species', 'sex', 'sex_type'])  # Features

Split the data into training and testing sets

In [23]:
# Split into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x_df, x_df, test_size=.25, random_state=42)

To optimize the features used, we can define a function that intakes a model and iteratively tries different feature sets.

In [ ]:
def test_model_accuracy(model: BaseEstimator, x_df: pd.DataFrame, feature_set: list, target_df: pd.DataFrame) -> tuple[float, BaseEstimator]:
    '''
    Helper function to assess a given model using given feature set and target dataframe
    Returns the accuracy and model
    '''
    temp_feature_df: pd.DataFrame = x_df[feature_set]  # Get the dataframe of just selected features
    # Split into testing and training
    x_train, x_test, y_train, y_test = train_test_split(temp_feature_df, target_df, test_size=.25, random_state=42)
    # Fit the data to the model and evaluate
    model.fit(x_train, y_train)
    return model.score(x_test, y_test), model  # Return the model accuracy and the trained model

def test_features(features: list, x_df: pd.DataFrame, target_df: pd.DataFrame, model: BaseEstimator = KNeighborsClassifier) -> BaseEstimator:
    ''' 
    Helper function to iteratively check all set of features, 
    printing the best set of features and returning the best model
    NOTE: This could be computational expensive, and NOT optimal
    '''
    best_accuracy: float = 0  # Tracker for the best accuracy
    best_model  # Tracker for the best model
    best_feature_set: list[str]  # Tracker for the best feature set
    assert isinstance(model, BaseEstimator)
    assert features  # Ensure that features exist
    assert target_df  # Ensure that the target df exists
    assert x_df  # Ensure that the x df exists
    
    # Iterate through all possible feature combinations, testing each
    for i in range(len(features)):
        cur_feature_set: list = [i]
        accuracy, cur_model = test_model_accuracy(model, x_df, features[i], target_df)
        best_accuracy = accuracy, best_model = cur_model, best_feature_set = \
            cur_feature_set if accuracy > best_accuracy else best_accuracy
        for j in range(len(features) - i):
            cur_feature_set.append(j)
            accuracy, cur_model = test_model_accuracy(model, x_df, cur_feature_set, target_df)
            best_accuracy = accuracy, best_model = cur_model, best_feature_set = \
                cur_feature_set if accuracy > best_accuracy else best_accuracy
    
    # Print the best accuracy and return the optimal model
    print(f"The best accuracy was {best_accuracy} with the feature set of {best_feature_set}")
    return best_model+

## References (Problem 10)

- [Pandas Online Documentation - Missing Data](https://pandas.pydata.org/docs/user_guide/10min.html#missing-data)
- [Pandas Online Documentation - Where Method](https://pandas.pydata.org/docs/user_guide/indexing.html#the-where-method-and-masking)